# Stage 3: Run a handler on toy data
> Run a complete handler on a small made-up seawater dataset, one step at a time, from the provider's CSV file to the MARIS NetCDF and CSV files

This page is a worked example. It runs a complete handler on a small seawater dataset, from the provider's CSV file to the MARIS NetCDF file and the CSV file that the MARIS database imports. The data is made up, but each of its problems comes from real provider data. Each section of the page fixes one of them.

Open `nbs/onboarding/toy-handler.ipynb` in VS Code, and run its cells one at a time from the top. Read each output before you run the next cell. In the terminal panel, start `claude` in `~/aai-ws/marisco` and ask it to read this notebook. Then ask it about anything you do not understand. For example:

- "What does `RemapCB` do with a nuclide name that is not in the lookup table?"
- "Show me the rows of the raw data that this step changes."
- "Where is `SanitizeLonLatCB` defined, and which handlers use it?"
- "What goes wrong if this step runs before the previous one?"

Change a cell if you want to try something. When you finish, `git restore nbs/onboarding/toy-handler.ipynb` discards your changes.

## The raw data

The provider, the made-up Toy Sea Monitoring Programme, measured Cs-137, Sr-90 and H-3 in seawater at five stations in the Baltic Sea in summer 2023. At each station it took one sample near the surface and one near the bottom. The CSV file has one row for each measurement.

In [ ]:
from fastcore.all import *
import pandas as pd, numpy as np
from marisco.callbacks import PerGroupCB, Transformer, RenameColumnsCB, RemapCB, SanitizeLonLatCB, AddSampleIDCB, EncodeTimeCB
from marisco.metadata import GlobAttrsFeeder, BboxCB, DepthRangeCB, TimeRangeCB, KeyValuePairCB
from marisco.encoders import NetCDFEncoder
from marisco.configs import get_lut
from marisco.nc2csv import to_csv

In [ ]:
fname = '../files/csv/toy-sea.csv'
raw = pd.read_csv(fname)
raw

,Station,Sample,Date,Latitude,Longitude,Depth (m),Nuclide,Activity,Uncertainty,Unit
0,TS01,TS01-S,28/06/2023,55.25,15.98,1,Cs-137,31.2,2.1,Bq/m3
1,TS01,TS01-S,28/06/2023,55.25,15.98,1,Sr-90,7.4,0.6,Bq/m3
2,TS01,TS01-S,28/06/2023,55.25,15.98,1,H-3,2.1,0.3,Bq/L
3,TS01,TS01-B,28/06/2023,55.25,15.98,45,Cs-137,35.8,2.4,Bq/m3
4,TS01,TS01-B,28/06/2023,55.25,15.98,45,Sr-90,<5.0,NaN,Bq/m3
5,TS01,TS01-B,28/06/2023,55.25,15.98,45,H-3,1.9,0.3,Bq/L
6,TS02,TS02-S,29/06/2023,55.50,18.50,1,Cs-137,28.4,1.9,Bq/m3
7,TS02,TS02-S,29/06/2023,55.50,18.50,1,Sr-90,6.9,0.6,Bq/m3
8,TS02,TS02-S,29/06/2023,55.50,18.50,1,H-3,2.4,0.3,Bq/L
9,TS02,TS02-B,29/06/2023,55.50,18.50,70,Cs-137,33.1,2.2,Bq/m3


MARIS cannot use this file as it is:

- The column names are the provider's, not the MARIS names.
- Dates are written day first: `03/07/2023` is 3 July.
- Some nuclides have two names: `Cs-137` and `137Cs`, `H-3` and `Tritium`.
- A value below the detection limit is written as text in the `Activity` column, as in `<5.0`.
- H-3 is in Bq/L. MARIS stores seawater activities in Bq/m3.
- Sample `TS05-S` has longitude 1904, which is not a position on Earth.

## How a handler is built

A handler reads the raw data into a dict of pandas DataFrames, with one DataFrame for each sample type group: `SEAWATER`, `BIOTA`, `SEDIMENT` or `SUSPENDED_MATTER`. This dataset has only `SEAWATER`. A `Transformer` then runs a list of callbacks on the DataFrames, in order. Each callback fixes one problem. Its docstring, the sentence in quotes under its `class` line, describes the fix. The `Transformer` collects these sentences in `tfm.logs`, and the NetCDF file stores them for the data provider to read.

Column names follow one rule. An uppercase name, such as `NUCLIDE` or `VALUE`, is a MARIS column, and the encoder writes it to the NetCDF file. A lowercase name, such as `activity`, is a working column. Callbacks use it, and the encoder ignores it.

`run` reads the raw file and runs the callbacks you give it. Each section below runs the pipeline again from the raw data, with one more callback than the section before.

In [ ]:
def run(*cbs):
    "Read the raw CSV, run the callbacks `cbs` on it, and return the `SEAWATER` DataFrame"
    return Transformer({'SEAWATER': pd.read_csv(fname)}, cbs=list(cbs))()['SEAWATER']

## Rename the columns

`RenameColumnsCB` renames the provider's columns and drops every column that `lut_cols` does not list. A column that is already correct gets its MARIS name. A column that a later step must still fix gets a lowercase working name.

In [ ]:
lut_cols = {'Station': 'STATION', 'Sample': 'SMP_ID_PROVIDER', 'Date': 'date', 'Latitude': 'LAT', 'Longitude': 'LON',
    'Depth (m)': 'SMP_DEPTH', 'Nuclide': 'nuclide', 'Activity': 'activity', 'Uncertainty': 'UNC', 'Unit': 'unit'}

In [ ]:
df = run(RenameColumnsCB(lut_cols))
df.head()

,STATION,SMP_ID_PROVIDER,date,LAT,LON,SMP_DEPTH,nuclide,activity,UNC,unit
0,TS01,TS01-S,28/06/2023,55.25,15.98,1,Cs-137,31.2,2.1,Bq/m3
1,TS01,TS01-S,28/06/2023,55.25,15.98,1,Sr-90,7.4,0.6,Bq/m3
2,TS01,TS01-S,28/06/2023,55.25,15.98,1,H-3,2.1,0.3,Bq/L
3,TS01,TS01-B,28/06/2023,55.25,15.98,45,Cs-137,35.8,2.4,Bq/m3
4,TS01,TS01-B,28/06/2023,55.25,15.98,45,Sr-90,<5.0,NaN,Bq/m3


## Parse the dates

Without a format, pandas guesses one. It reads `03/07/2023` month first, as 7 March:

In [ ]:
pd.to_datetime('03/07/2023')

Timestamp('2023-03-07 00:00:00')

`ParseToyDateCB` gives pandas the format, day first. No shared callback in `marisco.callbacks` reads this format. A handler defines its own callback when it needs one, and names it after the provider or after what it does. The [callback inventory](../reference/callback-inventory.ipynb) lists the shared callbacks and those of each handler.

In [ ]:
class ParseToyDateCB(PerGroupCB):
    "Parse the day-first `date` strings (dd/mm/yyyy) into a `TIME` column"
    def each_grp(self, grp, df, tfm): df['TIME'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

In [ ]:
df = run(RenameColumnsCB(lut_cols), ParseToyDateCB())
test_eq(df.loc[df.STATION=='TS04', 'TIME'].iloc[0], pd.Timestamp('2023-07-03'))
df[['STATION', 'date', 'TIME']].drop_duplicates()

,STATION,date,TIME
0,TS01,28/06/2023,2023-06-28
6,TS02,29/06/2023,2023-06-29
12,TS03,30/06/2023,2023-06-30
18,TS04,03/07/2023,2023-07-03
24,TS05,04/07/2023,2023-07-04


## Map nuclide names to MARIS identifiers

MARIS identifies each nuclide by a number from its nuclide lookup table. `get_lut` reads the table. These are the numbers of the three nuclides in this dataset:

In [ ]:
lut = get_lut('NUCLIDE')
{k: lut[k] for k in ['cs137', 'sr90', 'h3']}

{'cs137': 33, 'sr90': 12, 'h3': 1}

`NUCLIDE_LUT` maps each name that the provider uses to one of these numbers. `RemapCB` writes the number to a new `NUCLIDE` column. It writes 0, "not available", for a name that is not in the lookup, and the test below checks that there is none. A real dataset can have dozens of names. [Reconcile nomenclatures](../how-to/reconcile-nomenclature.ipynb) shows how to match them with fuzzy matching instead of by hand.

In [ ]:
NUCLIDE_LUT = {'Cs-137': 33, '137Cs': 33, 'Sr-90': 12, 'H-3': 1, 'Tritium': 1}

In [ ]:
df = run(RenameColumnsCB(lut_cols), ParseToyDateCB(), RemapCB(NUCLIDE_LUT, 'NUCLIDE', 'nuclide'))
test_eq((df.NUCLIDE==0).sum(), 0)
df[['nuclide', 'NUCLIDE']].value_counts()

nuclide  NUCLIDE
Sr-90    12         10
Cs-137   33          6
H-3      1           6
137Cs    33          4
Tritium  1           4
Name: count, dtype: int64

## Split detection limits from values

MARIS stores a measurement in two columns. `VALUE` holds the number, and `DL` says what the number is. These are the codes of the `DL` lookup table:

In [ ]:
get_lut('DL')

{'Not applicable': -1,
 'Not available': 0,
 'Detected value': 1,
 'Detection limit': 2,
 'Not detected': 3,
 'Derived': 4}

A value written `<5.0` means that the activity is below the detection limit of 5.0. `SplitDetectionLimitCB` stores it as `VALUE` 5.0 with `DL` 2, and every other value with `DL` 1. The provider gives no uncertainty for these rows.

In [ ]:
class SplitDetectionLimitCB(PerGroupCB):
    "Split `activity` into `VALUE` and a `DL` flag: 2 (detection limit) for values written as `<x`, else 1 (detected value)"
    def each_grp(self, grp, df, tfm):
        df['DL'] = np.where(df['activity'].str.startswith('<'), 2, 1)
        df['VALUE'] = df['activity'].str.lstrip('<').astype(float)

In [ ]:
df = run(RenameColumnsCB(lut_cols), ParseToyDateCB(), RemapCB(NUCLIDE_LUT, 'NUCLIDE', 'nuclide'), SplitDetectionLimitCB())
below = df[df.DL==2]
test_eq(len(below), 3)
below[['SMP_ID_PROVIDER', 'nuclide', 'activity', 'VALUE', 'UNC', 'DL']]

,SMP_ID_PROVIDER,nuclide,activity,VALUE,UNC,DL
4,TS01-B,Sr-90,<5.0,5.0,NaN,2
11,TS02-B,H-3,<1.0,1.0,NaN,2
16,TS03-B,Sr-90,<5.0,5.0,NaN,2


## Convert Bq/L to Bq/m3

MARIS stores seawater activities in Bq/m3, which is unit 1 in the unit lookup table. One cubic metre holds 1000 litres. An activity of 2.1 Bq/L is therefore 2100 Bq/m3. `ConvertToBqM3CB` multiplies `VALUE` and `UNC` by 1000 in the rows in Bq/L, and then sets `UNIT` to 1 in every row. A detection limit is in `VALUE` too, and the callback converts it with the other values.

In [ ]:
class ConvertToBqM3CB(PerGroupCB):
    "Convert values and uncertainties reported in Bq/L to Bq/m3 (x1000), the MARIS unit for seawater"
    def each_grp(self, grp, df, tfm):
        df.loc[df['unit']=='Bq/L', ['VALUE', 'UNC']] *= 1000
        df['UNIT'] = 1

In [ ]:
df = run(RenameColumnsCB(lut_cols), ParseToyDateCB(), RemapCB(NUCLIDE_LUT, 'NUCLIDE', 'nuclide'), SplitDetectionLimitCB(),
    ConvertToBqM3CB())
h3 = df[df.NUCLIDE==1]
test_eq(h3.VALUE.iloc[0], 2100)
test_eq(h3.UNC.iloc[0], 300)
h3[['SMP_ID_PROVIDER', 'activity', 'unit', 'VALUE', 'UNC', 'UNIT']].head()

,SMP_ID_PROVIDER,activity,unit,VALUE,UNC,UNIT
2,TS01-S,2.1,Bq/L,2100.0,300.0,1
5,TS01-B,1.9,Bq/L,1900.0,300.0,1
8,TS02-S,2.4,Bq/L,2400.0,300.0,1
11,TS02-B,<1.0,Bq/L,1000.0,NaN,1
14,TS03-S,2.2,Bq/L,2200.0,300.0,1


## Drop impossible positions

`SanitizeLonLatCB` is a shared callback from `marisco.callbacks`, and every handler uses it. It drops a row whose latitude is outside -90 to 90 or whose longitude is outside -180 to 180. It also drops a row at (0, 0), which providers often use for an unknown position. With `verbose=True` it prints the number of rows it drops:

In [ ]:
df = run(RenameColumnsCB(lut_cols), ParseToyDateCB(), RemapCB(NUCLIDE_LUT, 'NUCLIDE', 'nuclide'), SplitDetectionLimitCB(),
    ConvertToBqM3CB(), SanitizeLonLatCB(verbose=True))
test_eq(len(df), 27)
assert 'TS05-S' not in df.SMP_ID_PROVIDER.values

The "SEAWATER" group contains 3 data points with unrealistic LON or LAT values.


:::{.callout-important}
## FEEDBACK TO DATA PROVIDER
Sample `TS05-S` has longitude 1904. The bottom sample from the same station, `TS05-B`, has longitude 19.04, and the surface sample has probably lost its decimal point. The handler drops the three measurements of `TS05-S` and does not correct the longitude, because only the provider knows the real position. Please check the position of `TS05-S`.
:::

## Complete the pipeline

Two more shared callbacks complete the pipeline. `AddSampleIDCB` numbers the rows in `SMP_ID` and converts `SMP_ID_PROVIDER` to text for the NetCDF file. It runs after the last callback that drops rows. `EncodeTimeCB` converts `TIME` to seconds since 1 January 1970, the form that the NetCDF file stores.

`get_cbs` returns all the callbacks of the pipeline, in order. Every real handler defines a `get_cbs` function like this one.

In [ ]:
def get_cbs():
    "Callbacks, in pipeline order, that turn the raw toy data into MARIS-standard DataFrames"
    return [  # chkstyle: ignore-node
        RenameColumnsCB(lut_cols),
        ParseToyDateCB(),
        RemapCB(NUCLIDE_LUT, 'NUCLIDE', 'nuclide'),
        SplitDetectionLimitCB(),
        ConvertToBqM3CB(),
        SanitizeLonLatCB(),
        AddSampleIDCB(col_provider='SMP_ID_PROVIDER'),
        EncodeTimeCB(),
    ]

This time the cell keeps the `Transformer` in `tfm`, because the next sections use its logs. The table shows only the MARIS columns, the ones the encoder writes:

In [ ]:
tfm = Transformer({'SEAWATER': pd.read_csv(fname)}, cbs=get_cbs())
out = tfm()['SEAWATER']
out[[c for c in out.columns if c.isupper()]]

,STATION,SMP_ID_PROVIDER,LAT,LON,SMP_DEPTH,UNC,TIME,NUCLIDE,DL,VALUE,UNIT,SMP_ID
0,TS01,TS01-S,55.25,15.98,1,2.1,1687910400,33,1,31.2,1,1
1,TS01,TS01-S,55.25,15.98,1,0.6,1687910400,12,1,7.4,1,2
2,TS01,TS01-S,55.25,15.98,1,300.0,1687910400,1,1,2100.0,1,3
3,TS01,TS01-B,55.25,15.98,45,2.4,1687910400,33,1,35.8,1,4
4,TS01,TS01-B,55.25,15.98,45,NaN,1687910400,12,2,5.0,1,5
5,TS01,TS01-B,55.25,15.98,45,300.0,1687910400,1,1,1900.0,1,6
6,TS02,TS02-S,55.50,18.50,1,1.9,1687996800,33,1,28.4,1,7
7,TS02,TS02-S,55.50,18.50,1,0.6,1687996800,12,1,6.9,1,8
8,TS02,TS02-S,55.50,18.50,1,300.0,1687996800,1,1,2400.0,1,9
9,TS02,TS02-B,55.50,18.50,70,2.2,1687996800,33,1,33.1,1,10


In [ ]:
tfm.logs

['Rename variables to MARIS standard names, keeping only renamed columns',
 'Parse the day-first `date` strings (dd/mm/yyyy) into a `TIME` column',
 "Remap values from 'nuclide' to 'NUCLIDE' for groups: all.",
 'Split `activity` into `VALUE` and a `DL` flag: 2 (detection limit) for values written as `<x`, else 1 (detected value)',
 'Convert values and uncertainties reported in Bq/L to Bq/m3 (x1000), the MARIS unit for seawater',
 'Drop rows with invalid longitude & latitude values. Convert `,` separator to `.` separator',
 'Assign 1-based sequential SMP_ID; optionally cast a provider ID column to str for NetCDF VLEN compatibility',
 'Encode time as seconds since epoch']

## Write the NetCDF file

`GlobAttrsFeeder` builds the global attributes of the NetCDF file. `BboxCB`, `DepthRangeCB` and `TimeRangeCB` compute the area, depths and dates that the data covers. `KeyValuePairCB` adds the keywords and the logs. A real handler also adds the bibliographic record of the dataset, from Zotero or INIS. The toy dataset has none.

In [ ]:
attrs = GlobAttrsFeeder(tfm.dfs, cbs=[BboxCB(), DepthRangeCB(), TimeRangeCB(),
    KeyValuePairCB('keywords', 'Toy Sea, seawater, Cs-137, Sr-90, H-3'),
    KeyValuePairCB('publisher_postprocess_logs', ', '.join(tfm.logs))])()
attrs

{'geospatial_lat_min': '55.25',
 'geospatial_lat_max': '57.8',
 'geospatial_lon_min': '15.98',
 'geospatial_lon_max': '20.1',
 'geospatial_bounds': 'POLYGON ((15.98 55.25, 20.1 55.25, 20.1 57.8, 15.98 57.8, 15.98 55.25))',
 'geospatial_vertical_max': '90',
 'geospatial_vertical_min': '1',
 'time_coverage_start': '2023-06-28T00:00:00',
 'time_coverage_end': '2023-07-04T00:00:00',
 'keywords': 'Toy Sea, seawater, Cs-137, Sr-90, H-3',
 'publisher_postprocess_logs': "Rename variables to MARIS standard names, keeping only renamed columns, Parse the day-first `date` strings (dd/mm/yyyy) into a `TIME` column, Remap values from 'nuclide' to 'NUCLIDE' for groups: all., Split `activity` into `VALUE` and a `DL` flag: 2 (detection limit) for values written as `<x`, else 1 (detected value), Convert values and uncertainties reported in Bq/L to Bq/m3 (x1000), the MARIS unit for seawater, Drop rows with invalid longitude & latitude values. Convert `,` separator to `.` separator, Assign 1-based sequent

`NetCDFEncoder` writes the file to `_data/output/` in your marisco folder. Git ignores `_data/`, as `.gitignore` lists it.

In [ ]:
dest = Path('../../_data/output/toy-sea.nc')
dest.parent.mkdir(parents=True, exist_ok=True)
NetCDFEncoder(tfm.dfs, dest_fname=dest, global_attrs=attrs).encode()

## Export the MARIS CSV

`to_csv` converts the NetCDF file to one CSV file for each group, in the format that the MARIS database imports. `marisco-export` does the same from the terminal. The CSV uses the database's column names, and writes the `DL` code as `=` or `<`.

In [ ]:
csv = pd.read_csv(to_csv(dest)[0])
csv

,detection,latitude,longitude,nuclide_id,sampdepth,samplabcode,station,begperiod,uncertaint,unit_id,activity,samptype_id
0,=,55.25,15.98,33,1.0,TS01-S,TS01,2023-06-28,2.1,1,31.2,1
1,=,55.25,15.98,12,1.0,TS01-S,TS01,2023-06-28,0.6,1,7.4,1
2,=,55.25,15.98,1,1.0,TS01-S,TS01,2023-06-28,300.0,1,2100.0,1
3,=,55.25,15.98,33,45.0,TS01-B,TS01,2023-06-28,2.4,1,35.8,1
4,<,55.25,15.98,12,45.0,TS01-B,TS01,2023-06-28,NaN,1,5.0,1
5,=,55.25,15.98,1,45.0,TS01-B,TS01,2023-06-28,300.0,1,1900.0,1
6,=,55.50,18.50,33,1.0,TS02-S,TS02,2023-06-29,1.9,1,28.4,1
7,=,55.50,18.50,12,1.0,TS02-S,TS02,2023-06-29,0.6,1,6.9,1
8,=,55.50,18.50,1,1.0,TS02-S,TS02,2023-06-29,300.0,1,2400.0,1
9,=,55.50,18.50,33,70.0,TS02-B,TS02,2023-06-29,2.2,1,33.1,1


## Trace one measurement

Choose a measurement in the raw file, find it in the CSV, and check each of its values by hand. On a real dataset, this check finds mistakes that no test finds. Here is the H-3 measurement of sample `TS02-B`, which the provider reported as `<1.0` Bq/L:

In [ ]:
raw[(raw.Sample=='TS02-B') & (raw.Nuclide=='H-3')]

,Station,Sample,Date,Latitude,Longitude,Depth (m),Nuclide,Activity,Uncertainty,Unit
11,TS02,TS02-B,29/06/2023,55.5,18.5,70,H-3,<1.0,NaN,Bq/L


In the CSV it is a detection limit (`<`) of 1000 Bq/m3, with nuclide 1 (H-3) and unit 1 (Bq/m3):

In [ ]:
row = csv[(csv.samplabcode=='TS02-B') & (csv.nuclide_id==1)]
test_eq(row.activity.item(), 1000)
test_eq(row.detection.item(), '<')
row

,detection,latitude,longitude,nuclide_id,sampdepth,samplabcode,station,begperiod,uncertaint,unit_id,activity,samptype_id
11,<,55.5,18.5,1,70.0,TS02-B,TS02,2023-06-29,NaN,1,1000.0,1


## From this page to a real handler

A real handler in `nbs/handlers/` has the same steps as this page, and adds what the toy handler leaves out:

- Its `load_data` function downloads the raw data, often from Zenodo, and returns the dict of DataFrames.
- `#| export` directives put its callbacks and lookup tables into the `marisco` package.
- Its `encode` function runs the pipeline and writes the NetCDF file. `marisco-ingest` calls it.
- Its `status` tells `marisco-ingest` whether it can encode the dataset.
- Its global attributes include the bibliographic record of the dataset.

The [JOIS handler](../handlers/jois.ipynb) is a short real handler, with seawater only. Read it next, and find each step of this page in it.

## Check your understanding

Stage 3 is complete when you have run every cell, traced a second measurement of your choice from the raw file to the CSV, and can answer these questions:

- Why is `date` lowercase after renaming, while `LAT` is uppercase?
- What does `RemapCB` write for a nuclide name that is not in `NUCLIDE_LUT`?
- Why does `AddSampleIDCB` run after `SanitizeLonLatCB`?
- The provider sends a new file in which Cs-137 is also written `CS137`. Which line of this notebook do you change?
- Why does the handler drop sample `TS05-S` instead of correcting its longitude to 19.04?